# Disciplyn: train all 14 behaviour models on Kaggle

**Before running:**
1. Upload `disciplyn_kaggle_code.zip` as a Kaggle **Dataset** (Datasets → New Dataset). Kaggle unzips it automatically.
2. In this notebook, open **Add Input** and select that dataset.
3. Settings: **Accelerator = None (CPU)**. No GPU or internet is needed; Kaggle already has every library this uses.
4. Click **Save Version → Save & Run All (Commit)**. It keeps running after you close the browser.
   Full run time is about 20–35 minutes, well inside Kaggle's 12-hour limit.
5. Afterwards, open the version → **Output** → download `disciplyn_out_models_and_reports.zip`.

The script is crash-safe. If anything stops it, run the same cell again and it resumes from the last finished model.

In [ ]:
import os, shutil, glob, sys, subprocess
# locate the uploaded code (any dataset folder containing run_all.py)
hits = glob.glob('/kaggle/input/**/run_all.py', recursive=True)
assert hits, 'run_all.py not found - did you add the dataset as input?'
src = os.path.dirname(hits[0])
CODE = '/kaggle/working/code'
shutil.rmtree(CODE, ignore_errors=True)
shutil.copytree(src, CODE, ignore=shutil.ignore_patterns('data', 'models', 'run_output*'))
os.chdir(CODE)
print('code copied from', src); print(sorted(os.listdir(CODE)))
import sklearn, pandas, numpy, joblib
print('python', sys.version.split()[0], '| sklearn', sklearn.__version__, '| pandas', pandas.__version__, '| cpus', os.cpu_count())

## 1. Smoke test (about 2 minutes). If this cell fails, stop and fix the problem before the long run

In [ ]:
r = subprocess.run([sys.executable, '-W', 'ignore', 'run_all.py', '--quick', '--fresh'], capture_output=True, text=True)
print(r.stdout[-4000:]); print(r.stderr[-3000:])
assert 'FINISHED: 14/14' in r.stdout, 'smoke test failed - see log above'
print('SMOKE TEST PASSED')

## 2. Full run: 10,000 traders × 60 days (about 1.9M trades), 14 models

In [ ]:
# streams the log live; safe to re-run (resumes from checkpoints)
p = subprocess.Popen([sys.executable, '-W', 'ignore', 'run_all.py'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    if 'simulated' in line or 'features:' in line:
        if not any(k in line for k in ('2,500/', '5,000/', '7,500/', '10,000/', '5,000 traders', '10,000 traders')):
            continue
    print(line, end='')
print('exit code', p.wait())

## 3. Results

In [ ]:
import pandas as pd
from IPython.display import Image, display
OUT = '/kaggle/working/disciplyn_out'
res = pd.read_csv(f'{OUT}/reports/test_metrics.csv')
display(res[['model','algorithm','accuracy','precision','recall','f1','roc_auc','train_roc_auc','gap','fit']])
print(res[['accuracy','f1','roc_auc']].mean().round(4))
display(Image(f'{OUT}/reports/fit_check.png')); display(Image(f'{OUT}/reports/test_metrics.png')); display(Image(f'{OUT}/reports/roc_curves.png'))

In [ ]:
# keep the Output tab small: remove the raw 470 MB trades file and the quick-test folder
for f in ['/kaggle/working/disciplyn_out/data/trades.csv']:
    if os.path.exists(f): os.remove(f)
shutil.rmtree('/kaggle/working/disciplyn_out_quick', ignore_errors=True)
for f in glob.glob('/kaggle/working/*_quick_models_and_reports.zip'): os.remove(f)
print(os.listdir('/kaggle/working'))